# 2.0 – Statistical Analysis & Predictive Modeling
**DATA 205 | Feraol Abera | Spring 2026**

- Q4: Does time of day affect warning vs. citation likelihood? → Logistic Regression
- Q6: Do demographics relate to violation type? → Chi-Square
- Q7: What factors influence whether a search is conducted? → Logistic Regression
- Q8: Does search conducted relate to stop outcome? → Chi-Square
- Q9: Which violations have the highest arrest probability? → Bar Chart
- Q10: Can we predict stop outcome from key variables? → Random Forest

In [ ]:
!pip install scikit-learn --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
from scipy.stats import chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import joblib
from google.colab import drive

drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/data205-project'
IMG  = os.path.join(BASE, 'analysis/images')
os.makedirs(IMG, exist_ok=True)
os.makedirs(os.path.join(BASE, 'models'), exist_ok=True)

BLUE = '#1B4F72'
RED  = '#E74C3C'
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 130,
                     'axes.spines.top': False,
                     'axes.spines.right': False})

In [ ]:
tv = pd.read_csv(
    os.path.join(BASE, 'data/processed/traffic_violations_clean.csv'),
    parse_dates=['date_of_stop'],
    low_memory=False
)

## Q4 — Warning vs. Citation: Does Time of Day Matter? (Logistic Regression)

In [ ]:
q4 = tv[tv['violation_type'].isin(['Warning', 'Citation'])].copy()
q4['is_citation'] = (q4['violation_type'] == 'Citation').astype(int)
q4['dow_num'] = pd.Categorical(
    q4['day_of_week'],
    categories=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
).codes

bool_feats = ['search_conducted', 'alcohol', 'accident']
feat_cols  = ['hour', 'dow_num'] + [c for c in bool_feats if c in q4.columns]

q4[feat_cols] = q4[feat_cols].apply(pd.to_numeric, errors='coerce')
q4 = q4.dropna(subset=feat_cols + ['is_citation'])

X = q4[feat_cols]
y = q4['is_citation']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [ ]:
pipe_q4 = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('clf',    LogisticRegression(max_iter=500, random_state=42))
])
pipe_q4.fit(X_train, y_train)
y_pred_q4 = pipe_q4.predict(X_test)

print(classification_report(y_test, y_pred_q4, target_names=['Warning', 'Citation']))

In [ ]:
coefs = pd.Series(pipe_q4.named_steps['clf'].coef_[0], index=feat_cols).sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
colors = [RED if v > 0 else BLUE for v in coefs.values]
ax.barh(coefs.index, coefs.values, color=colors, alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Q4 — Logistic Regression: Warning vs. Citation', fontsize=14, fontweight='bold')
ax.set_xlabel('Coefficient (log-odds)')
plt.tight_layout()
plt.savefig(os.path.join(IMG, 'q4_logreg_coefficients.png'), bbox_inches='tight')
plt.show()

## Q6 — Demographics × Violation Type (Chi-Square)

In [ ]:
for demo in ['race', 'gender']:
    ct = pd.crosstab(tv[demo], tv['violation_type'])
    chi2, p, dof, _ = chi2_contingency(ct)
    print(f'{demo.upper()} × violation_type: Chi²={chi2:,.2f}, df={dof}, p={p:.4e}')

In [ ]:
top_races = tv['race'].value_counts().head(5).index

ct_pct = (
    tv[tv['race'].isin(top_races)]
    .pipe(lambda d: pd.crosstab(d['race'], d['violation_type'], normalize='index'))
)

ct_pct.plot(kind='bar', stacked=True, figsize=(10, 5),
            colormap='Blues', edgecolor='white', linewidth=0.5)
plt.title('Q6 — Violation Type by Race (row %)', fontsize=14, fontweight='bold')
plt.xlabel(None)
plt.ylabel('Proportion')
plt.xticks(rotation=20, ha='right')
plt.legend(title='Violation Type', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(IMG, 'q6_violation_by_race.png'), bbox_inches='tight')
plt.show()

In [ ]:
ct_gender = (
    tv.pipe(lambda d: pd.crosstab(d['gender'], d['violation_type'], normalize='index'))
)

ct_gender.plot(kind='bar', stacked=True, figsize=(7, 4),
               colormap='Blues', edgecolor='white', linewidth=0.5)
plt.title('Q6 — Violation Type by Gender (row %)', fontsize=14, fontweight='bold')
plt.xlabel(None)
plt.ylabel('Proportion')
plt.xticks(rotation=0)
plt.legend(title='Violation Type', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(IMG, 'q6_violation_by_gender.png'), bbox_inches='tight')
plt.show()

## Q7 — What Predicts Whether a Search is Conducted? (Logistic Regression)

In [ ]:
q7 = tv.copy()
q7['search_conducted'] = pd.to_numeric(q7['search_conducted'], errors='coerce')
q7 = q7.dropna(subset=['search_conducted'])

q7['vtype_enc']  = LabelEncoder().fit_transform(q7['violation_type'].fillna('Unknown'))
q7['gender_enc'] = LabelEncoder().fit_transform(q7['gender'].fillna('Unknown'))

feats_q7 = ['hour', 'vtype_enc', 'gender_enc', 'alcohol', 'accident']
q7[feats_q7] = q7[feats_q7].apply(pd.to_numeric, errors='coerce')
q7 = q7.dropna(subset=feats_q7)

X7 = q7[feats_q7]
y7 = q7['search_conducted'].astype(int)

X7_train, X7_test, y7_train, y7_test = train_test_split(
    X7, y7, test_size=0.25, random_state=42, stratify=y7
)

In [ ]:
pipe_q7 = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('clf',    LogisticRegression(max_iter=500, class_weight='balanced', random_state=42))
])
pipe_q7.fit(X7_train, y7_train)
y7_pred = pipe_q7.predict(X7_test)

print(classification_report(y7_test, y7_pred, target_names=['No Search', 'Search']))

In [ ]:
coefs_q7 = pd.Series(pipe_q7.named_steps['clf'].coef_[0], index=feats_q7).sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
colors7 = [RED if v > 0 else BLUE for v in coefs_q7.values]
ax.barh(coefs_q7.index, coefs_q7.values, color=colors7, alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Q7 — Logistic Regression: Search Conducted Predictors', fontsize=14, fontweight='bold')
ax.set_xlabel('Coefficient (log-odds)')
plt.tight_layout()
plt.savefig(os.path.join(IMG, 'q7_search_conducted_logreg.png'), bbox_inches='tight')
plt.show()

## Q8 — Search Conducted × Stop Outcome (Chi-Square)

In [ ]:
q8 = tv.dropna(subset=['search_conducted', 'violation_type']).copy()
q8['search_conducted'] = pd.to_numeric(q8['search_conducted'], errors='coerce')
q8 = q8.dropna(subset=['search_conducted'])
q8['search_conducted'] = q8['search_conducted'].astype(int)

ct8 = pd.crosstab(q8['search_conducted'], q8['violation_type'])
chi2_8, p_8, dof_8, _ = chi2_contingency(ct8)

print(f'Chi² = {chi2_8:,.2f},  df = {dof_8},  p = {p_8:.4e}')
ct8

In [ ]:
ct8_pct = ct8.div(ct8.sum(axis=1), axis=0)

ct8_pct.index = ['No Search', 'Search']
ct8_pct.plot(kind='bar', stacked=True, figsize=(7, 4),
             colormap='Blues', edgecolor='white', linewidth=0.5)
plt.title('Q8 — Violation Type by Search Conducted (row %)', fontsize=14, fontweight='bold')
plt.xlabel(None)
plt.ylabel('Proportion')
plt.xticks(rotation=0)
plt.legend(title='Violation Type', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(IMG, 'q8_search_vs_outcome.png'), bbox_inches='tight')
plt.show()

## Q9 — Arrest Probability by Violation Type

In [ ]:
q9 = tv.dropna(subset=['arrest_type', 'violation_type']).copy()

# Arrests = anything that isn't a marked patrol release
q9['arrested'] = (~q9['arrest_type'].str.upper().str.contains(
    'MARKED PATROL|A - MARKED|NO ARREST', na=True
)).astype(int)

arrest_prob = (
    q9.groupby('violation_type')['arrested']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'arrest_rate', 'count': 'total'})
    .query('total >= 100')
    .sort_values('arrest_rate', ascending=False)
    .head(10)
)

arrest_prob

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(
    arrest_prob.index[::-1],
    arrest_prob['arrest_rate'][::-1] * 100,
    color=RED, alpha=0.85
)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{w:.1f}%', va='center', fontsize=10)
ax.set_title('Q9 — Arrest Rate by Violation Type (top 10)', fontsize=14, fontweight='bold')
ax.set_xlabel('Arrest Rate (%)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
plt.tight_layout()
plt.savefig(os.path.join(IMG, 'q9_arrest_rate_by_violation.png'), bbox_inches='tight')
plt.show()

## Q10 — Random Forest: Predict Stop Outcome

In [ ]:
top_outcomes = tv['violation_type'].value_counts().head(4).index

rf_df = tv[tv['violation_type'].isin(top_outcomes)].copy()

# Encode categoricals
rf_df['dow_num']     = pd.Categorical(
    rf_df['day_of_week'],
    categories=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
).codes
rf_df['period_num']  = pd.Categorical(
    rf_df['time_period'],
    categories=['Morning','Afternoon','Evening','Night']
).codes
rf_df['gender_enc']  = LabelEncoder().fit_transform(rf_df['gender'].fillna('Unknown'))
rf_df['race_enc']    = LabelEncoder().fit_transform(rf_df['race'].fillna('Unknown'))

le_y = LabelEncoder()
rf_df['target'] = le_y.fit_transform(rf_df['violation_type'])

feat_rf = ['hour', 'month', 'dow_num', 'period_num',
           'gender_enc', 'race_enc',
           'search_conducted', 'alcohol', 'accident', 'work_zone']
feat_rf = [c for c in feat_rf if c in rf_df.columns]

rf_df[feat_rf] = rf_df[feat_rf].apply(pd.to_numeric, errors='coerce')
rf_df = rf_df.dropna(subset=feat_rf + ['target'])

# Sample for speed
rf_df = rf_df.sample(min(40000, len(rf_df)), random_state=42)

X_rf = rf_df[feat_rf]
y_rf = rf_df['target']

X_tr, X_te, y_tr, y_te = train_test_split(
    X_rf, y_rf, test_size=0.25, random_state=42, stratify=y_rf
)
print(f'Train: {len(X_tr):,}  |  Test: {len(X_te):,}  |  Classes: {le_y.classes_}')

In [ ]:
rf_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('clf',    RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=10,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipe.fit(X_tr, y_tr)
y_pred_rf = rf_pipe.predict(X_te)

print(classification_report(y_te, y_pred_rf, target_names=le_y.classes_))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_te, y_pred_rf,
    display_labels=le_y.classes_,
    cmap='Blues', ax=ax, colorbar=False
)
ax.set_title('Q10 — Confusion Matrix: Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(IMG, 'q10_confusion_matrix.png'), bbox_inches='tight')
plt.show()

In [ ]:
importances = pd.Series(
    rf_pipe.named_steps['clf'].feature_importances_,
    index=feat_rf
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
importances.plot(kind='bar', color=BLUE, alpha=0.85, ax=ax)
ax.set_title('Q10 — Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
ax.set_xlabel('Feature')
ax.set_ylabel('Importance')
ax.set_xticklabels(importances.index, rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(IMG, 'q10_feature_importance.png'), bbox_inches='tight')
plt.show()

In [ ]:
joblib.dump(rf_pipe, os.path.join(BASE, 'models/random_forest_stop_outcome.joblib'))